<a href="https://colab.research.google.com/github/KrAzad0/Monte-Carlo/blob/main/monte_carlo_inference_of_the_cosmological_constant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy import integrate

# -----------------------------
# Constants and helpers
# -----------------------------
c_km_s = 299792.458  # km/s
MPC_IN_PC = 1e6

def E_of_z(z, OmL):
    # Flat LCDM: OmM = 1 - OmL
    return np.sqrt((1.0 - OmL) * (1 + z)**3 + OmL)

def dL_Mpc(z, H0, OmL):
    # d_L in Mpc
    integrand = lambda zp: 1.0 / E_of_z(zp, OmL)
    # handle scalar or vector z
    z = np.atleast_1d(z)
    dL = np.empty_like(z, dtype=float)
    for i, zi in enumerate(z):
        I = integrate.quad(integrand, 0.0, zi, epsabs=1e-8, epsrel=1e-8)[0]
        dL[i] = (1 + zi) * (c_km_s / H0) * I
    return dL

def mu_theory(z, H0, OmL, Moff):
    # Moff absorbs SN absolute magnitude + H0 zero point.
    # Using conventional mu = 5 log10(dL/Mpc) + 25 + Moff
    dL = dL_Mpc(z, H0, OmL)
    return 5.0 * np.log10(dL) + 25.0 + Moff

# -----------------------------
# Synthetic data (toggle use_real_data)
# -----------------------------
use_real_data = False
if use_real_data:
    # Load your CSV with columns: z, mu, sigma
    data = np.genfromtxt('my_sne.csv', delimiter=',', names=True)
    z_data = data['z']; mu_data = data['mu']; sig_data = data['sigma']
else:
    # Synthesize a modest Hubble diagram
    rng = np.random.default_rng(42)
    true_params = dict(H0=70.0, OmL=0.70, Moff=-19.3)  # (note: Moff acts like an intercept)
    z_data = np.sort(rng.uniform(0.01, 0.8, size=60))
    sig_data = 0.12 * np.ones_like(z_data)  # 0.12 mag intrinsic+meas scatter
    mu_clean = mu_theory(z_data, **true_params)
    mu_data = mu_clean + rng.normal(0, sig_data)

# -----------------------------
# Log-likelihood + log-prior
# -----------------------------
def loglike(theta):
    H0, OmL, Moff = theta
    # Basic sanity
    if not (40.0 < H0 < 100.0):
        return -np.inf
    if not (0.0 < OmL < 1.0):
        return -np.inf
    # likelihood
    mu_model = mu_theory(z_data, H0, OmL, Moff)
    chi2 = np.sum((mu_data - mu_model)**2 / sig_data**2)
    return -0.5 * chi2 - 0.5 * np.sum(np.log(2*np.pi*sig_data**2))

def logprior(theta):
    H0, OmL, Moff = theta
    # Priors: H0 ~ N(70,10^2); OmL ~ U(0,1); Moff ~ N(-19.3, 0.5^2)
    lp = 0.0
    lp += -0.5 * ((H0 - 70.0)/10.0)**2 - np.log(10.0*np.sqrt(2*np.pi))
    if 0.0 < OmL < 1.0:
        lp += 0.0  # uniform(0,1)
    else:
        return -np.inf
    lp += -0.5 * ((Moff + 19.3)/0.5)**2 - np.log(0.5*np.sqrt(2*np.pi))
    return lp

def logpost(theta):
    lp = logprior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = loglike(theta)
    return lp + ll

# -----------------------------
# Metropolis–Hastings MCMC
# -----------------------------
rng = np.random.default_rng(123)
ndim = 3
nsteps = 30000
start = np.array([70.0, 0.5, -19.0])  # initial guess: (H0, OmL, Moff)
cov = np.array([3.0, 0.05, 0.05])**2    # proposal variances

chain = np.zeros((nsteps, ndim))
lp_cur = logpost(start)
theta = start.copy()
accept = 0

for i in range(nsteps):
    prop = theta + rng.normal(0, np.sqrt(cov))
    lp_prop = logpost(prop)
    if np.log(rng.uniform()) < (lp_prop - lp_cur):
        theta = prop
        lp_cur = lp_prop
        accept += 1
    chain[i] = theta

acc_rate = accept / nsteps
burn = nsteps // 2
post = chain[burn:]

H0_samp = post[:,0]
OmL_samp = post[:,1]
Moff_samp = post[:,2]

# Compute the cosmological constant Λ in SI from samples:
# Λ = 3 H0^2 ΩΛ / c^2, with H0 in s^{-1}. Convert km/s/Mpc -> s^{-1}.
KM_PER_M = 1e-3
MPC_IN_M = 3.0856775814913673e22
H0_s_inv = (H0_samp * (KM_PER_M) ) / MPC_IN_M  # (km/s)/Mpc -> s^{-1}
Lambda_samp = 3.0 * (H0_s_inv**2) * OmL_samp / ( (c_km_s*KM_PER_M)**2 )

def summarize(x):
    q = np.percentile(x, [16,50,84])
    return q[1], (q[1]-q[0]), (q[2]-q[1])

H0_med, H0_m, H0_p   = summarize(H0_samp)
OmL_med, OmL_m, OmL_p = summarize(OmL_samp)
Lam_med, Lam_m, Lam_p = summarize(Lambda_samp)

print(f"Acceptance rate ~ {acc_rate:.2f}")
print(f"Omega_Lambda = {OmL_med:.3f} -{OmL_m:.3f} +{OmL_p:.3f}")
print(f"H0 = {H0_med:.1f} -{H0_m:.1f} +{H0_p:.1f} km/s/Mpc")
print(f"Lambda = {Lam_med:.3e} -{Lam_m:.3e} +{Lam_p:.3e} 1/m^2")


Acceptance rate ~ 0.14
Omega_Lambda = 0.660 -0.052 +0.051
H0 = 71.9 -6.3 +6.8 km/s/Mpc
Lambda = 1.202e-52 -2.374e-53 +2.741e-53 1/m^2
